In [2]:
import os

In [3]:
%pwd

'e:\\Replica\\wine_mlops\\research'

In [4]:
os.chdir('../')

In [5]:
%pwd

'e:\\Replica\\wine_mlops'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [7]:
from wine_quality.constants import *
from wine_quality.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH, 
        params_filepath = PARAMS_FILE_PATH, 
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir = Path(config.root_dir),
            train_data_path = Path(config.train_data_path),
            test_data_path = Path(config.test_data_path),
            model_name = config.model_name,
            
            alpha = params.alpha,
            l1_ratio = params.l1_ratio,
            target_column = schema.name
        )

        return model_trainer_config

In [9]:
import pandas as ps
import os
from wine_quality import logger
from sklearn.linear_model import ElasticNet
import joblib

In [10]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        logger.info(f"Training model {self.config.model_name}")

        train_data = ps.read_csv(self.config.train_data_path)
        test_data = ps.read_csv(self.config.test_data_path)

        logger.info(f"Train data shape: {train_data.shape}")
        logger.info(f"Test data shape: {test_data.shape}")

        X_train = train_data.drop(columns=[self.config.target_column], axis=1)
        y_train = train_data[self.config.target_column]
        X_test = test_data.drop(columns=[self.config.target_column], axis=1)
        y_test = test_data[self.config.target_column]

        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(X_train, y_train)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

        


In [11]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()

except Exception as e:
    logger.exception(e)
    raise e

[2025-05-01 08:17:30,026: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-05-01 08:17:30,029: INFO: common: yaml file: params.yaml loaded successfully]
[2025-05-01 08:17:30,032: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-05-01 08:17:30,034: INFO: common: created directory at: artifacts]
[2025-05-01 08:17:30,036: INFO: common: created directory at: artifacts/model_trainer]
[2025-05-01 08:17:30,037: INFO: 2208639206: Training model model.joblib]
[2025-05-01 08:17:30,050: INFO: 2208639206: Train data shape: (1279, 12)]
[2025-05-01 08:17:30,052: INFO: 2208639206: Test data shape: (320, 12)]
